# Modelling - Quick Implementation

In [1]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions
spark = DatabricksSession.builder.getOrCreate()
from sklearn.model_selection import train_test_split

## Feature Engineering

### Phase I
- Starting off with 5 simple features that can be traced to domain experience and eda done. They include: `Type_risk`, `Power`, `Value_vehicle`, `Seniority`, `Payment`
- For each of these features - implement the transforms for each e.g value vehicle - log transform it?
- Derive the exposure here - eda reveals its 1 year for policy period and the targets (`severity`) as `machine_learning.complete_dataset` layer
- Rename the target to make it clear to identify the targets

In [2]:
CATALOG = "commercial-insurance-underwriting-solution-catalog"
SEED = 42
policy = spark.table(f"`{CATALOG}`.silver.policy")
policy.show(5)

+---+-------------------+-----------------+-----------------+----------+--------------------+--------------------+---------+-----------------+------------+------------+-----+----------+-------+-------+----------------+-------------+----------------+----------------+---------+----+-------------+------------------+-----+-----------------+-------------+-------+---------+------+------+
| ID|Date_start_contract|Date_last_renewal|Date_next_renewal|Date_birth|Date_driving_licence|Distribution_channel|Seniority|Policies_in_force|Max_policies|Max_products|Lapse|Date_lapse|Payment|Premium|Cost_claims_year|N_claims_year|N_claims_history|R_Claims_history|Type_risk|Area|Second_driver|Year_matriculation|Power|Cylinder_capacity|Value_vehicle|N_doors|Type_fuel|Length|Weight|
+---+-------------------+-----------------+-----------------+----------+--------------------+--------------------+---------+-----------------+------------+------------+-----+----------+-------+-------+----------------+------------

In [3]:
feature_column_list = ['ID','Type_risk', 'Power', 'Value_vehicle', 'Seniority', 'Payment']
target_column = ['N_claims_year', 'Cost_claims_year']
dataset_columns_ordered = ['ID', 'Exposure', *feature_column_list[1:], 'Value_vehicle_log', 'Claims_Frequency', 'Claims_Severity', 'Loss_Cost']
basic_features_and_targets = (
    policy
    .select(feature_column_list+target_column)
    .withColumn('Value_vehicle_log', functions.log(functions.col('Value_vehicle').alias("Value_vehicle_log")))
    .withColumn('Exposure', functions.lit(1))
    .withColumn('Claims_Severity', functions.try_divide(target_column[1], target_column[0])) #try divide does cost_claims / number of claims and return null for cases where number of claims = 0
    .withColumnRenamed(target_column[0], 'Claims_Frequency')
    .withColumnRenamed(target_column[1], 'Loss_Cost')
    .select(dataset_columns_ordered))

In [4]:
basic_features_and_targets_lower_case = basic_features_and_targets.toDF(*[c.lower() for c in basic_features_and_targets.columns])
basic_features_and_targets_lower_case.show(5)

+---+--------+---------+-----+-------------+---------+-------+-----------------+----------------+---------------+---------+
| id|exposure|type_risk|power|value_vehicle|seniority|payment|value_vehicle_log|claims_frequency|claims_severity|loss_cost|
+---+--------+---------+-----+-------------+---------+-------+-----------------+----------------+---------------+---------+
|  1|       1|        1|   80|       7068.0|        4|      0|8.863332833439587|               0|           NULL|      0.0|
|  1|       1|        1|   80|       7068.0|        4|      0|8.863332833439587|               0|           NULL|      0.0|
|  1|       1|        1|   80|       7068.0|        4|      0|8.863332833439587|               0|           NULL|      0.0|
|  1|       1|        1|   80|       7068.0|        4|      0|8.863332833439587|               0|           NULL|      0.0|
|  2|       1|        1|   80|       7068.0|        4|      1|8.863332833439587|               0|           NULL|      0.0|
+---+---

In [3]:
## Write this back into a machine_learning schema as the complete dataset
CATALOG = "commercial-insurance-underwriting-solution-catalog"
SCHEMA = "machine_learning"
TABLE = "complete_dataset"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
basic_features_and_targets_lower_case.write.mode('Overwrite').saveAsTable(f"`{CATALOG}`.{SCHEMA}.{TABLE}")

### Phase II
- Each row in source is a policy per year. It is possible not to have claims on the first 3 year and in the fourth year for example. With this understanding, splitting will be done at the policy level using the ID
- From EDA, the proportion of zeros for target variables is hight also there are very few but large claims as well, the goal of this split is to ensure those are representative. As such we would be ensuring proportional split across zeros, body and large claims
- Split complete dataset into train, dev, test split, maintain the proportions in all the target variables
- Do some sense checks e.g. no id not duplicated in different split i.e one id must always be in one split and not 2 or more.

In [4]:
complete_dataset = spark.table(f"`{CATALOG}`.{SCHEMA}.{TABLE}")
complete_dataset.show(5)

+---+--------+---------+-----+-------------+---------+-------+-----------------+----------------+---------------+---------+
| id|exposure|type_risk|power|value_vehicle|seniority|payment|value_vehicle_log|claims_frequency|claims_severity|loss_cost|
+---+--------+---------+-----+-------------+---------+-------+-----------------+----------------+---------------+---------+
|  1|       1|        1|   80|       7068.0|        4|      0|8.863332833439587|               0|           NULL|      0.0|
|  1|       1|        1|   80|       7068.0|        4|      0|8.863332833439587|               0|           NULL|      0.0|
|  1|       1|        1|   80|       7068.0|        4|      0|8.863332833439587|               0|           NULL|      0.0|
|  1|       1|        1|   80|       7068.0|        4|      0|8.863332833439587|               0|           NULL|      0.0|
|  2|       1|        1|   80|       7068.0|        4|      1|8.863332833439587|               0|           NULL|      0.0|
+---+---

In [5]:
loss_cost_95th_percentile = (
    complete_dataset
    .filter(functions.col('loss_cost') > 0)
    .select(functions.percentile_approx('loss_cost', 0.95).alias('p95'))
    .first()['p95']
)

In [6]:
## Group the dataset by id and then get out ids and the maximum loss cost categories
grouped = (
    complete_dataset
    .groupBy('id')
    .agg(
        functions.max('loss_cost').alias('max_loss_cost'),
    ))

id_lookup_bucket = (
    grouped.withColumn(
        'bucket',
        functions.when(functions.col('max_loss_cost')==0, 'zero')
        .when(functions.col('max_loss_cost')<= loss_cost_95th_percentile, 'body')
        .otherwise('tail'))
)

In [7]:
id_lookup_df = (id_lookup_bucket.toPandas())
train_ids, remainder_ids = train_test_split(
    id_lookup_df,
    test_size=0.35
    ,random_state=SEED
    ,stratify=id_lookup_df['bucket']
)

val_ids, test_ids = train_test_split(
     remainder_ids,
    test_size=0.4
    ,random_state=SEED
    ,stratify= remainder_ids['bucket']
)

In [14]:
train_ids_sdf, val_ids_sdf, test_ids_sdf = spark.createDataFrame(train_ids), spark.createDataFrame(val_ids), spark.createDataFrame(test_ids)
train_sdf, val_sdf, test_sdf = complete_dataset.join(train_ids_sdf.select('id'), on='id', how='inner'), complete_dataset.join(val_ids_sdf.select('id'), on='id', how='inner'), complete_dataset.join(test_ids_sdf.select('id'), on='id', how='inner')

In [12]:
## quick validation
assert(train_sdf.count() + val_sdf.count() + test_sdf.count() == complete_dataset.count())

In [19]:
# Write it back to
split_tables = {
    train_sdf:'train',
    val_sdf : 'val',
    test_sdf : 'test'
}

for split, split_name in split_tables.items():
    split.write.mode('Overwrite').saveAsTable(f"`{CATALOG}`.{SCHEMA}.{split_name}")

## Modelling

## Evaluation